In [26]:
! pip install kagglehub

In [27]:
import pandas as pd 
import numpy as np
import kagglehub
import os

In [28]:
# Descargar la última version del dataset de Kaggle
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

# Revisar que archivos contiene
print(os.listdir(path))

Path to dataset files: C:\Users\Ivanna\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2
['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


## Comprensión de los datos

In [29]:
import pandas as pd
import os

customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
geolocation = pd.read_csv(os.path.join(path, "olist_geolocation_dataset.csv"))
orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
order_items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
order_payments = pd.read_csv(os.path.join(path, "olist_order_payments_dataset.csv"))
order_reviews = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv"))
products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))
sellers = pd.read_csv(os.path.join(path, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(
    os.path.join(path, "product_category_name_translation.csv")
)

## Dimensión de las bases de datos

In [30]:
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Orders": orders,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for nombre, df in datasets.items():
    print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")

Customers: 99441 filas, 5 columnas
Geolocation: 1000163 filas, 5 columnas
Orders: 99441 filas, 8 columnas
Order Items: 112650 filas, 7 columnas
Order Payments: 103886 filas, 5 columnas
Order Reviews: 99224 filas, 7 columnas
Products: 32951 filas, 9 columnas
Sellers: 3095 filas, 4 columnas
Category Translation: 71 filas, 2 columnas


## Duplicados

In [31]:
for nombre, df in datasets.items():
    print(f"{nombre}: {df.duplicated().sum()} duplicados")

Customers: 0 duplicados
Geolocation: 261831 duplicados
Orders: 0 duplicados
Order Items: 0 duplicados
Order Payments: 0 duplicados
Order Reviews: 0 duplicados
Products: 0 duplicados
Sellers: 0 duplicados
Category Translation: 0 duplicados


En este caso estuvimos viendo los duplicados en las base de datos en dónde pudimos encontrar que la única con estos fué Geolocation con 261831 duplicados, pero estos se estarán viendo más adelante en la parte de la exploración de los datos. 

## Verificar las llaves primarias

In [32]:
print(customers["customer_id"].is_unique)
print(orders["order_id"].is_unique)
print(products["product_id"].is_unique)
print(sellers["seller_id"].is_unique)
print(category_translation["product_category_name"].is_unique)

True
True
True
True
True


Cómo podemos ver las llaves primarias de las bases de datos son las siguientes:

* customer_id para la base de datos Customers
* order_id para la base de datos Orders
* product_id para la base de datos Products
* geolocation_id para la base de datos Geolocation. 

La cuales nos ayudarán a conectar las bases de datos de acuerdo a nuestro objetivo de negocio y poder llegar a una solución.

## Revisar los tipos de datos

In [33]:
for nombre, df in datasets.items():
    print(f"\n{nombre}")
    print(df.dtypes)


Customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Orders
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

Order Items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype:

Se verificaron los tipos de datos de las nueve tablas del conjunto de datos. Los identificadores se encuentran almacenados como variables de tipo object, mientras que las variables numéricas presentan tipos int64 y float64, lo cual es consistente con la naturaleza de la variable. Se identificó que las variables correspondientes a fechas y horas en las tablas orders, order_items y order_reviews se encuentran almacenadas como object; por tanto, durante la fase de preparación de los datos se convertirán al tipo datetime para facilitar el análisis temporal. No se identificaron inconsistencias relevantes en los demás tipos de datos.

# Completitud de los datos

#  Consistencia de los datos

# Trazabilidad de los datos